# 🛡️ [AI SIEM] Random Forest 기반 Shadow API & IDOR 탐지 모델 학습
본 노트북은 **Wazuh 로그의 30초 시간창(Time Window) 특징값**을 바탕으로,
L4 방화벽(FortiGate) 및 WAF를 우회하는 지능형 웹 침투(PentAGI) 공격을 실시간 탐지하는 AI 모델을 학습하고 평가합니다.

### 📊 핵심 목표 (계획서 §6 달성 여부 확인)
1. **탐지율 (Recall)**: $\ge 80\%$
2. **정상 오탐률 (FPR)**: $< 2.5\%$ (최대 $5.0\%$)
3. **최초 탐지 시간 (TTD)**: 60초 이내
4. **특징 중요도 (Feature Importance)**: 공격자 식별에 기여한 핵심 행위 분석

## 1. 데이터 파일 업로드 (`features_dataset.csv`)
내 컴퓨터의 `d:\Project\team_one\ml\data\features_dataset.csv` 파일을 아래 셀을 실행하여 업로드하세요.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 코랩 파일 업로드 위젯
from google.colab import files
print("[*] 'features_dataset.csv' 파일을 선택하여 업로드해주세요:")
uploaded = files.upload()

csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)
print(f"\n[+] 성공적으로 로드되었습니다: {csv_filename}")
print(f"총 샘플 수: {len(df)}개 창 (정상: {sum(df['label']==0)}개, 공격: {sum(df['label']==1)}개)")
df.head()

## 2. 특징값(Features)과 라벨(Label) 분리 및 분포 확인

In [ ]:
META_COLS = {'session_id', 'window_idx', 'client_ip', 'window_start', 'label'}
feature_cols = [c for c in df.columns if c not in META_COLS]

print(f"총 {len(feature_cols)}개의 행위 피처:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i}. {col}")

# 공격 vs 정상 피처 평균 비교
print("\n[정상 vs 공격 평균 행동 패턴 비교]")
df.groupby('label')[feature_cols].mean().rename(index={0: '정상(Human)', 1: '공격(PentAGI)'}).T

## 3. 시계열 순서 기준 Train / Test 분할
시간 순서에 따라 각 세션의 **앞쪽 75%를 학습(Train), 뒤쪽 25%를 검증(Test)**으로 분할하여 데이터 누수(Data Leakage)를 방지합니다.

In [ ]:
test_frac = 0.25
train_rows, test_rows = [], []

for sid, group in df.groupby("session_id"):
    group = group.sort_values("window_start")
    n = len(group)
    n_test = max(1, round(n * test_frac)) if n > 1 else 0
    n_train = n - n_test
    train_rows.append(group.iloc[:n_train])
    test_rows.append(group.iloc[n_train:])

df_train = pd.concat(train_rows, ignore_index=True)
df_test = pd.concat(test_rows, ignore_index=True)

X_train, y_train = df_train[feature_cols], df_train['label'].astype(int)
X_test, y_test = df_test[feature_cols], df_test['label'].astype(int)

print(f"학습용(Train): {len(df_train)}개 (정상: {sum(y_train==0)}, 공격: {sum(y_train==1)})")
print(f"검증용(Test) : {len(df_test)}개 (정상: {sum(y_test==0)}, 공격: {sum(y_test==1)})")

## 4. Random Forest 모델 학습
앙상블 기법인 Random Forest를 사용하며, 정상/공격 불균형을 고려해 `class_weight='balanced'`를 적용합니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)
print("🎉 Random Forest 모델 학습이 완료되었습니다!")

## 5. 모델 평가 및 핵심 4대 지표 산출
혼동 행렬(Confusion Matrix)과 계획서의 **탐지율(Recall), 오탐률(FPR), 탐지시간(TTD)**을 검증합니다.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

recall = tp / (tp + fn) if (tp + fn) else 0.0
fpr = fp / (fp + tn) if (fp + tn) else 0.0

print("=" * 60)
print(" [계획서 §6 핵심 성과 4대 지표 달성 결과]")
print("=" * 60)
print(f"① 탐지율 (Recall)       : {recall * 100:.2f}%  (목표 >= 80%)")
print(f"② 정상 오탐률 (FPR)     : {fpr * 100:.2f}%  (목표 < 2.5%)")

# 최초 탐지 시간(TTD)
dft = df_test.copy()
dft['_ts'] = pd.to_datetime(dft['window_start'])
dft['_pred'] = y_pred
ttds = []
for sid, g in dft[dft['label'] == 1].groupby('session_id'):
    g = g.sort_values('_ts')
    hits = g[g['_pred'] == 1]
    if not hits.empty:
        ttds.append((hits['_ts'].iloc[0] - g['_ts'].iloc[0]).total_seconds())

if ttds:
    print(f"③ 최초 탐지 시간 (TTD)  : {np.median(ttds):.0f}초 (목표 30~60초 이내)")
print("=" * 60)

# 상세 분류 리포트
print("\n[상세 리포트]")
print(classification_report(y_test, y_pred, target_names=['정상(0)', '공격(1)'], digits=4))

## 6. 시각화: 혼동 행렬(Heatmap) 및 특징 중요도(Feature Importance)

In [ ]:
plt.figure(figsize=(14, 5))

# 1. 혼동 행렬 히트맵
plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['정상(0)', '공격(1)'], 
            yticklabels=['정상(0)', '공격(1)'], cbar=False)
plt.title('Confusion Matrix (혼동 행렬)', fontsize=13)
plt.xlabel('예측값 (Predicted)')
plt.ylabel('실제값 (Actual)')

# 2. 특징 중요도 막대 차트
plt.subplot(1, 2, 2)
imp = sorted(zip(feature_cols, rf_model.feature_importances_), key=lambda t: t[1])
names = [c for c, _ in imp]
vals = [v for _, v in imp]
plt.barh(names, vals, color='#2563EB')
plt.title('Feature Importance (특징 중요도 순위)', fontsize=13)
plt.xlabel('중요도 점수')

plt.tight_layout()
plt.show()

## 7. 학습된 모델 저장 및 다운로드
학습된 모델을 `.joblib` 파일로 저장하고 내 컴퓨터로 다운로드합니다.

In [ ]:
import joblib

model_filename = 'pentagi_rf_model.joblib'
joblib.dump({'model': rf_model, 'features': feature_cols}, model_filename)
print(f"[+] 모델 저장 완료: {model_filename}")

# 내 컴퓨터로 다운로드
files.download(model_filename)